# Structured Output

I can ask the model to respond in JSON and I can even specify the exact schema that I want the response in. This use case is different from function calling where the model's output will be piped to some function that will use the structured output. In this scenario I want the end-user output to be structured, e.g., maybe because my client app consumes this output in some way.

In [6]:
from dotenv import load_dotenv
from openai import OpenAI
import pydantic as pt
from utils import LLM

In [2]:
load_dotenv()

True

In [3]:
client = OpenAI()

In [4]:
class CalendarEvent(pt.BaseModel):
    name: str
    date: str
    # date: datetime
    participants: list[str]

In [5]:
CalendarEvent.model_json_schema()

{'properties': {'name': {'title': 'Name', 'type': 'string'},
  'date': {'title': 'Date', 'type': 'string'},
  'participants': {'items': {'type': 'string'},
   'title': 'Participants',
   'type': 'array'}},
 'required': ['name', 'date', 'participants'],
 'title': 'CalendarEvent',
 'type': 'object'}

In [7]:
completion = client.beta.chat.completions.parse(
    model=LLM.PRE_FAST_MINI,
    messages=[
        {"role": "developer", "content": "Extract the event information."},
        {
            "role": "user",
            "content": "Alice and Bob are going to a science fair this Friday.",
        },
    ],
    response_format=CalendarEvent,
)
event_response = completion.choices[0].message

In [8]:
event_response

ParsedChatCompletionMessage[CalendarEvent](content='{"name":"Science Fair","date":"2023-10-20","participants":["Alice","Bob"]}', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None, parsed=CalendarEvent(name='Science Fair', date='2023-10-20', participants=['Alice', 'Bob']))

```python
ParsedChatCompletionMessage[CalendarEvent](
    content='{"name":"Science Fair","date":"2023-10-20","participants":["Alice","Bob"]}', 
    refusal=None, 
    role='assistant', 
    annotations=[], 
    audio=None, 
    function_call=None, 
    tool_calls=None, 
    parsed=CalendarEvent(
        name='Science Fair', 
        date='2023-10-20', 
        participants=['Alice', 'Bob']
    )
)
```

!!! The model seems to be stuck in 2023. !!!

If the request could not be fulfilled for safety reasons, then the `refusal` property will be set. Otherwise I can get the JSON representation out from the `content` property, or get the instantiated object from the `parsed` property.

In [9]:
event = event_response.parsed if not event_response.refusal else None
event

CalendarEvent(name='Science Fair', date='2023-10-20', participants=['Alice', 'Bob'])

Only a limited number of JSON types are supported. E.g., when I change type of `date` to `datetime`, it throws an error. Here are the JSON types that are supported -
  * String
  * Number
  * Boolean
  * Integer
  * Object
  * Array
  * Enum
  * anyOf

I can union these with `null` to indicate optional fields. However, to the API I must always say that all the fields are required. If I am using Pydantic, then this is automatically taken care of.

Instead of giving the Pydantic object as the schema reference, I can just directly pass the schema in the API as shown in the next example. Here I must ensure that `additionalProperties` is not set to `true`, the default is `false`, all the defined properties are set in `required`.

In [10]:
class Cookie(pt.BaseModel):
    flavor: str
    calories: int | None

In [11]:
Cookie.model_json_schema()

{'properties': {'flavor': {'title': 'Flavor', 'type': 'string'},
  'calories': {'anyOf': [{'type': 'integer'}, {'type': 'null'}],
   'title': 'Calories'}},
 'required': ['flavor', 'calories'],
 'title': 'Cookie',
 'type': 'object'}

In [12]:
response_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "cookie_response",
        "schema": {
            "type": "object",
            "title": "Cookie",
            "properties": {
                "flavor": {"title": "Flavor", "type": "string"},
                "calories": {
                    "title": "Calories",
                    "anyOf": [{"type": "integer"}, {"type": "null"}],
                },
            },
            "required": ["flavor", "calories"],
            "additionalProperties": False,
        },
    },
}

In [13]:
completion = client.beta.chat.completions.parse(
    model=LLM.PRE_FAST_MINI,
    messages=[
        {
            "role": "developer",
            "content": "Extract cookie information about the cookie that AP likes.",
        },
        {
            "role": "user",
            "content": "AP loves to eat Oatmeal Raisin cookies! At 180 calories, these are lower than the Chocolate Chip cookies that Anika likes.",
        },
    ],
    response_format=response_format,  # type: ignore
)
cookie_response = completion.choices[0].message
cookie_response

ParsedChatCompletionMessage[NoneType](content='{"flavor":"Oatmeal Raisin","calories":180}', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None, parsed=None)

```python
ParsedChatCompletionMessage[NoneType](
    content='{"flavor":"Oatmeal Raisin","calories":180}', 
    refusal=None, 
    role='assistant', 
    annotations=[], 
    audio=None, 
    function_call=None, 
    tool_calls=None, 
    parsed=None
)
```
Because I didn't provide a Pydantic object, the `parsed` property is not set. I need to convert the JSON in the `content` property to my object.

In [14]:
cookie = (
    Cookie.model_validate_json(cookie_response.content)
    if cookie_response and cookie_response.content and not cookie_response.refusal
    else None
)
cookie

Cookie(flavor='Oatmeal Raisin', calories=180)

As a best practice I should include instructions on how to handle "bad" input, which can be anything that results in a response that does not confirm to the given object schema. My instructions can be to return None or a specific error message, etc.

I can also stream the output -

In [15]:
class Step(pt.BaseModel):
    explanation: str
    output: str


class MathReasoning(pt.BaseModel):
    steps: list[Step]
    final_answer: str

In [16]:
messages = [
    {
        "role": "developer",
        "content": "You are a helpful math tutor. Guide the user through the solution step by step.",
    },
    {"role": "user", "content": "How can I solve 8x + 7 = -23"},
]

In [17]:
with client.beta.chat.completions.stream(
    model=LLM.PRE_FAST_MINI, messages=messages  # type: ignore
) as stream:
    for event in stream:
        if event.type == "content.delta":
            if event.parsed:
                print("content.delta.parsed: ", event.parsed)
        elif event.type == "content.done":
            print("content.done")
        elif event.type == "error":
            print("Error in stream: ", event.error)

final_completion = stream.get_final_completion()
print("Final completion: ", final_completion)

content.done
Final completion:  ParsedChatCompletion[NoneType](id='chatcmpl-BNBDjFC5IKEMugyrrwsttOXWNZvqj', choices=[ParsedChoice[NoneType](finish_reason='stop', index=0, logprobs=None, message=ParsedChatCompletionMessage[NoneType](content='To solve the equation \\( 8x + 7 = -23 \\), follow these steps:\n\n1. **Isolate the term with the variable**: We want to isolate \\( 8x \\) on one side of the equation. Start by subtracting \\( 7 \\) from both sides.\n\n   \\[\n   8x + 7 - 7 = -23 - 7\n   \\]\n\n   Simplifying both sides gives:\n\n   \\[\n   8x = -30\n   \\]\n\n2. **Solve for \\( x \\)**: Next, we want to isolate \\( x \\) by dividing both sides by \\( 8 \\).\n\n   \\[\n   x = \\frac{-30}{8}\n   \\]\n\n   Simplifying the fraction, we can divide both the numerator and denominator by \\( 2 \\):\n\n   \\[\n   x = \\frac{-15}{4}\n   \\]\n\nSo the solution to the equation \\( 8x + 7 = -23 \\) is \n\n\\[\nx = -\\frac{15}{4}\n\\] \n\nor in decimal form \\( x = -3.75 \\).', refusal=None, ro